# 10d Evaluate and Refine Concepts

Build yearly abstract-prevalence signals and score them with fixed direct forecasts. Run this notebook during discovery rounds; edit `CURRENT_POPULATION`, `CANDIDATE_IDS`, and `PARENT_CHILD_REPLACEMENTS` after reviewing the development results. Confirmation pairs are not used to choose concepts.

In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "amr").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "amr").exists():
    raise RuntimeError(f"Could not find repository root from {Path.cwd()}")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from amr.pipeline_config import cfg, cfg_path, load_pipeline_config
from amr.pipelines.concepts import build_si_concept_paths, load_concept_corpus, corpus_preflight
from amr.si.concepts import (
    SEED_CONCEPTS, concept_catalog_hash, estimate_classification_cost,
    classify_concept_abstracts, build_yearly_concept_signals,
    rolling_origin_concept_forecasts, summarise_incremental_forecast_gains,
    propose_concept_children,
)

PIPELINE_CONFIG = load_pipeline_config(project_root=PROJECT_ROOT)
CONCEPT_PATHS = build_si_concept_paths(PROJECT_ROOT)
MODEL_NAME = str(cfg(PIPELINE_CONFIG, "llm.model", "gpt-5-nano"))

corpus = pd.read_parquet(CONCEPT_PATHS["corpus"])
si_df = pd.read_parquet(PROJECT_ROOT / "outputs" / "parquets" / "yearly_si.parquet")
catalog = json.loads(CONCEPT_PATHS["catalog"].read_text())
CATALOG_VERSION = concept_catalog_hash(catalog)
scores = pd.read_parquet(CONCEPT_PATHS["scores"])
valid = scores.loc[
    scores["catalog_version"].astype(str).eq(CATALOG_VERSION)
    & scores["model_name"].astype(str).eq(MODEL_NAME)
    & scores["status"].eq("ok")
]
expected_pmids = set(corpus.loc[corpus["abstract_available"], "pmid"].astype(str))
scored_pmids = set(valid["pmid"].astype(str))
missing_pmids = expected_pmids - scored_pmids
if missing_pmids:
    raise RuntimeError(f"Catalog scores are incomplete for {len(missing_pmids):,} abstracts. Resume notebook 10c first.")
signals = build_yearly_concept_signals(corpus, scores, catalog_version=CATALOG_VERSION, model_name=MODEL_NAME)
signals.to_parquet(CONCEPT_PATHS["signals"], index=False)


## Configure the discovery round

In [ ]:
CONCEPT_IDS = [str(concept["concept_id"]) for concept in catalog]
CURRENT_POPULATION = []  # concept IDs that currently survive the development round
CANDIDATE_IDS = CONCEPT_IDS  # initially compare all seed concepts
PARENT_CHILD_REPLACEMENTS = {}  # example: {"surveillance": ["genomic_surveillance", "hospital_surveillance"]}
model_sets = {f"concept:{concept_id}": [concept_id] for concept_id in CANDIDATE_IDS}
comparisons = {f"concept:{concept_id}": "baseline" for concept_id in CANDIDATE_IDS}
if CURRENT_POPULATION:
    model_sets["population"] = list(CURRENT_POPULATION)
    comparisons["population"] = "baseline"
    for candidate in CANDIDATE_IDS:
        if candidate not in CURRENT_POPULATION:
            name = f"population_plus:{candidate}"
            model_sets[name] = [*CURRENT_POPULATION, candidate]
            comparisons[name] = "population"
for parent_id, child_ids in PARENT_CHILD_REPLACEMENTS.items():
    model_sets[f"parent:{parent_id}"] = [parent_id]
    model_sets[f"children:{parent_id}"] = list(child_ids)
    comparisons[f"parent:{parent_id}"] = "baseline"
    comparisons[f"children:{parent_id}"] = f"parent:{parent_id}"


## Development rolling-origin forecasts

In [ ]:
split = pd.read_csv(CONCEPT_PATHS["pair_split"])
development_pairs = set(map(tuple, split.loc[split["split"].eq("development"), ["pathogen", "antibiotic"]].astype(str).to_numpy()))
forecasts = rolling_origin_concept_forecasts(
    si_df, signals, concept_ids=CONCEPT_IDS, model_sets=model_sets,
    development_pairs=development_pairs,
    fit_start_year=int(cfg(PIPELINE_CONFIG, "si.fit_start_year", 2000)),
    fit_end_year=int(cfg(PIPELINE_CONFIG, "si.fit_end_year", 2023)),
    lag_length=3, horizons=(1, 2, 3), min_prior_observations=10,
)
forecasts = forecasts.merge(split, on=["pathogen", "antibiotic"], how="left", validate="many_to_one")
forecasts = forecasts.loc[forecasts["split"].eq("development")].drop(columns="split")
forecasts.to_parquet(CONCEPT_PATHS["forecasts"], index=False)
pair_summary, population_summary = summarise_incremental_forecast_gains(forecasts, comparisons)
pair_summary.to_parquet(CONCEPT_PATHS["pair_summary"], index=False)
population_summary.to_parquet(CONCEPT_PATHS["population_summary"], index=False)
population_summary.sort_values("generality", ascending=False)


## Horizon-specific and concept-level results

In [ ]:
pair_summary.groupby(["model", "horizon"], as_index=False).agg(
    generality=("forecast_gain", "mean"),
    specialist_strength=("forecast_gain", lambda values: values.quantile(0.9)),
    pairs=("forecast_gain", "count"),
    forecasts=("forecasts", "sum"),
).sort_values(["model", "horizon"])
